# 📊 Running Evaluations

This notebook evaluates the trained models across three evaluation tracks.

## Three Evaluation Tracks

| Track | Purpose | Metrics |
|-------|---------|--------|
| **A. `prepared_diagnostics`** | Diagnostics on offline holdout data | Answer accuracy, tool match, grounding score |
| **B. `tau_episodes`** | Run official τ-Knowledge episodes | task_success_rate, pass^k |
| **C. `retention`** | Verify existing capability retention | ARC-Challenge accuracy, retention_delta_pp |

### Key Distinctions

- **A is lab diagnostics**: Synthetic single-turn accuracy; this is **not** the official τ score.
- **B is official evaluation**: Runs real episodes and calls deployed model endpoints.
- **C is the forgetting test**: Disables KB/RAG and runs general benchmarks.

### Comparison Variants

```
Base × {no_knowledge, rag}
LoRA × {no_knowledge, rag}
OSFT × {no_knowledge, rag}
```

All variants use the same business tools and simulator conditions.

> ⚠️ The `tau_episodes` track requires deployed model endpoints.  
> If no endpoints are available, only `prepared_diagnostics` and `retention` will run.

In [ ]:
"""Run prepared_diagnostics — offline holdout evaluation."""

import os
import sys
import subprocess
from pathlib import Path

from rhoai_model_training_lab.config import load_env, load_eval_config, PROJECT_ROOT

load_env()

eval_config = load_eval_config()
output_dir = PROJECT_ROOT / eval_config["general"]["output_dir"]
output_dir.mkdir(parents=True, exist_ok=True)

print("=" * 70)
print("📊 Track A: prepared_diagnostics (offline holdout)")
print("=" * 70)

diag_config = eval_config["tracks"]["prepared_diagnostics"]
if not diag_config.get("enabled", True):
    print("⏭️ prepared_diagnostics is disabled, skipping.")
else:
    print(f"  Holdout path: {diag_config['holdout_path']}")
    print(f"  Evaluation types: {diag_config['types']}")
    print(f"  matched_context: {diag_config.get('matched_context', False)}")
    print()

    # Run via CLI
    run_eval_script = PROJECT_ROOT / "scripts" / "run_eval.py"

    if run_eval_script.exists():
        cmd = [
            sys.executable, str(run_eval_script),
            "--track", "prepared_diagnostics",
            "--config", "configs/eval.yaml",
        ]
        print(f"  Command: {' '.join(cmd)}")
        result = subprocess.run(cmd, capture_output=True, text=True, cwd=str(PROJECT_ROOT))

        if result.returncode == 0:
            print("\n✅ prepared_diagnostics completed")
            print(result.stdout[-500:] if len(result.stdout) > 500 else result.stdout)
        else:
            print(f"\n❌ Execution failed (exit code: {result.returncode})")
            print(result.stderr[-500:] if len(result.stderr) > 500 else result.stderr)
    else:
        print("  ⚠️  run_eval.py script not found.")
        print("  Manual execution:")
        print("    python scripts/run_eval.py --track prepared_diagnostics --config configs/eval.yaml")

    print("\n  ℹ️  These results are labeled as 'τ-Knowledge derived lab diagnostics'.")
    print("     Synthetic single-turn accuracy ≠ official τ pass rate")

In [ ]:
"""Run tau_episodes smoke test (5 episodes, 1 trial)."""

print("=" * 70)
print("📊 Track B: tau_episodes (official τ-Knowledge episodes)")
print("=" * 70)

tau_config = eval_config["tracks"]["tau_episodes"]
if not tau_config.get("enabled", True):
    print("⏭️ tau_episodes is disabled, skipping.")
else:
    smoke_limit = tau_config.get("smoke_limit", 5)
    smoke_trials = tau_config.get("smoke_trials", 1)

    print(f"  Domain: {tau_config.get('domain', 'banking_knowledge')}")
    print(f"  Smoke limit: {smoke_limit} episodes, {smoke_trials} trials")
    print(f"  Variants: {[v['name'] for v in tau_config.get('variants', [])]}")
    print()

    # Check if model endpoints are available
    base_endpoint = os.environ.get("BASE_SERVING_ENDPOINT", "")
    if not base_endpoint:
        print("⚠️  Model serving endpoint is not configured.")
        print("   Please set BASE_SERVING_ENDPOINT, LORA_SERVING_ENDPOINT, OSFT_SERVING_ENDPOINT.")
        print("   Skipping smoke test.")
    else:
        run_eval_script = PROJECT_ROOT / "scripts" / "run_eval.py"
        if run_eval_script.exists():
            cmd = [
                sys.executable, str(run_eval_script),
                "--track", "tau_episodes",
                "--config", "configs/eval.yaml",
                "--limit", str(smoke_limit),
                "--trials", str(smoke_trials),
            ]
            print(f"  Command: {' '.join(cmd)}")
            print("  (This may take several minutes)\n")

            result = subprocess.run(cmd, capture_output=True, text=True, cwd=str(PROJECT_ROOT))

            if result.returncode == 0:
                print("\n✅ tau_episodes smoke test completed")
                print(result.stdout[-500:] if len(result.stdout) > 500 else result.stdout)
            else:
                print(f"\n❌ Execution failed (exit code: {result.returncode})")
                print(result.stderr[-500:] if len(result.stderr) > 500 else result.stderr)
        else:
            print("  ⚠️  run_eval.py script not found")
            print("  Manual execution:")
            print(f"    python scripts/run_eval.py --track tau_episodes --config configs/eval.yaml --limit {smoke_limit} --trials {smoke_trials}")

    print("\n  ℹ️  The smoke run is for path validation, not for benchmark performance conclusions.")
    print("     pass^k measures consistent success across repeated trials (different from pass@k).")

In [ ]:
"""Run retention benchmark (ARC-Challenge)."""

print("=" * 70)
print("📊 Track C: retention (existing capability retention)")
print("=" * 70)

retention_config = eval_config["tracks"]["retention"]
if not retention_config.get("enabled", True):
    print("⏭️ retention is disabled, skipping.")
else:
    benchmarks = retention_config.get("benchmarks", [])
    print(f"  Benchmarks: {[b['name'] for b in benchmarks]}")
    print(f"  KB/RAG disabled: {retention_config.get('disable_kb_rag', True)}")
    print(f"  Banking system prompt disabled: {retention_config.get('disable_banking_system_prompt', True)}")
    print()

    for bench in benchmarks:
        print(f"\n  📝 {bench['name']}:")
        print(f"     Dataset: {bench.get('dataset', 'N/A')}")
        print(f"     Subset: {bench.get('subset', 'N/A')}")
        print(f"     Sample size: {bench.get('sample_size', 'N/A')}")
        print(f"     Type: {bench.get('type', 'generated_answer')}")

    run_eval_script = PROJECT_ROOT / "scripts" / "run_eval.py"
    if run_eval_script.exists():
        cmd = [
            sys.executable, str(run_eval_script),
            "--track", "retention",
            "--config", "configs/eval.yaml",
        ]
        print(f"\n  Command: {' '.join(cmd)}")

        result = subprocess.run(cmd, capture_output=True, text=True, cwd=str(PROJECT_ROOT))

        if result.returncode == 0:
            print("\n✅ retention evaluation completed")
            print(result.stdout[-500:] if len(result.stdout) > 500 else result.stdout)
        else:
            print(f"\n❌ Execution failed (exit code: {result.returncode})")
            print(result.stderr[-500:] if len(result.stderr) > 500 else result.stderr)
    else:
        print("\n  Manual execution:")
        print("    python scripts/run_eval.py --track retention --config configs/eval.yaml")

    print("\n  ℹ️  retention_delta_pp = 100 × (adapted_accuracy - base_accuracy)")
    print("     A small sample from a single benchmark cannot prove forgetting elimination.")

In [ ]:
"""View per-task results."""

import json

print("=" * 70)
print("📋 View Per-Task Results")
print("=" * 70)

results_dir = output_dir

# Find result files
result_files = sorted(results_dir.glob("**/*.json"))
jsonl_files = sorted(results_dir.glob("**/*.jsonl"))

if not result_files and not jsonl_files:
    print("⚠️  No result files found.")
    print(f"   Results directory: {results_dir}")
    print("   Please run the evaluation tracks above first.")
else:
    print(f"Result files: {len(result_files)} JSON, {len(jsonl_files)} JSONL\n")

    # Load and display summary files
    for rf in result_files:
        if "summary" in rf.name or rf.name == "results.json":
            print(f"\n--- {rf.relative_to(results_dir)} ---")
            try:
                with open(rf) as f:
                    data = json.load(f)

                if isinstance(data, dict):
                    for key, value in data.items():
                        if isinstance(value, (int, float, str, bool)):
                            print(f"  {key}: {value}")
                        elif isinstance(value, dict) and len(value) <= 10:
                            print(f"  {key}:")
                            for k, v in value.items():
                                print(f"    {k}: {v}")
            except Exception as exc:
                print(f"  (Failed to load: {exc})")

    # Show per-task JSONL
    for jf in jsonl_files[:3]:
        print(f"\n--- {jf.relative_to(results_dir)} ---")
        try:
            with open(jf) as f:
                lines = f.readlines()[:5]
            for line in lines:
                rec = json.loads(line)
                task_id = rec.get("task_id", "?")
                success = rec.get("success", "?")
                error = rec.get("error", "")
                status = "✅" if success else "❌"
                print(f"  {status} {task_id}: success={success}", end="")
                if error:
                    print(f" error={error[:60]}")
                else:
                    print()
            if len(lines) >= 5:
                print(f"  ... (total {sum(1 for _ in open(jf))} rows)")
        except Exception as exc:
            print(f"  (Failed to load: {exc})")

In [ ]:
"""Log all evaluation results to MLflow."""

print("=" * 70)
print("📦 MLflow Evaluation Result Logging")
print("=" * 70)

mlflow_uri = os.environ.get("MLFLOW_TRACKING_URI", "")
experiment_name = os.environ.get("MLFLOW_EXPERIMENT_EVAL", "rhoai-model-training-lab-evaluation")

if not mlflow_uri:
    print("⚠️  MLFLOW_TRACKING_URI not set")
    print("   Local results have been saved.")
    print(f"   Results path: {output_dir}")
    print("\n   Manual upload:")
    print("   python scripts/log_eval_results.py --results-dir", str(output_dir))
else:
    # Use log_eval_results script if available
    log_script = PROJECT_ROOT / "scripts" / "log_eval_results.py"
    if log_script.exists():
        cmd = [
            sys.executable, str(log_script),
            "--results-dir", str(output_dir),
        ]
        print(f"  Running: {' '.join(cmd)}")
        result = subprocess.run(cmd, capture_output=True, text=True, cwd=str(PROJECT_ROOT))

        if result.returncode == 0:
            print("\n✅ MLflow logging completed")
            print(result.stdout[-300:] if len(result.stdout) > 300 else result.stdout)
        else:
            print(f"\n❌ MLflow logging failed (exit code: {result.returncode})")
            print(result.stderr[-300:] if len(result.stderr) > 300 else result.stderr)
    else:
        # Direct MLflow logging
        try:
            import mlflow

            mlflow.set_tracking_uri(mlflow_uri)
            mlflow.set_experiment(experiment_name)

            # Log each result file
            for rf in result_files:
                with mlflow.start_run(run_name=rf.stem) as run:
                    try:
                        with open(rf) as f:
                            data = json.load(f)
                        if isinstance(data, dict):
                            for k, v in data.items():
                                if isinstance(v, (int, float)):
                                    mlflow.log_metric(k, v)
                                elif isinstance(v, str):
                                    mlflow.log_param(k, v[:250])
                        mlflow.log_artifact(str(rf))
                        print(f"  ✅ {rf.name} → run_id: {run.info.run_id}")
                    except Exception as exc:
                        print(f"  ❌ {rf.name}: {exc}")

            print("\n✅ MLflow logging completed")
        except Exception as exc:
            print(f"\n❌ MLflow logging failed: {exc}")
            print("  Local results have been preserved.")

print("\nNext steps:")
print("  📓 08_compare_results.ipynb — Experiment Result Comparison")